<a href="https://colab.research.google.com/github/MaryamEltokhy/NHA-5-187/blob/main/notebooks/05_train_unet_MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Google Drive and project paths
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/BrainMRI_Data')
SPLIT_CSV = BASE / 'qc/split_v1.csv'
PROCESSED = Path('/content/processed')
OUT = BASE / 'runs/M1-T05_smoke_mlflow'
MLFLOW_DIR = BASE / 'mlruns'

OUT.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('SPLIT_CSV:', SPLIT_CSV)
print('PROCESSED:', PROCESSED)
print('MLFLOW_DIR:', MLFLOW_DIR)
assert SPLIT_CSV.exists(), f'Missing {SPLIT_CSV}. Run the existing data-preparation/split pipeline first.'

Mounted at /content/drive
BASE: /content/drive/MyDrive/BrainMRI_Data
SPLIT_CSV: /content/drive/MyDrive/BrainMRI_Data/qc/split_v1.csv
PROCESSED: /content/processed
MLFLOW_DIR: /content/drive/MyDrive/BrainMRI_Data/mlruns


In [ ]:
# 2. Install the packages needed for this partial implementation
!pip -q install 'monai>=1.4' mlflow nibabel pandas pyyaml tqdm scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.

## 3. Reuse the exact project code from the existing smoke-test notebook

The next cell recreates the same `src/bmts`, loader config, and `scripts/smoke_train.py` structure used by the uploaded smoke-test notebook. The only change is the MLflow instrumentation added to `smoke_train.py`.

In [ ]:
from pathlib import Path

CODE = Path('/content/repo')
FILES = {'src/bmts/__init__.py': '"""Brain MRI tumor system (bmts): DEPI graduation project package."""\n', 'src/bmts/common/__init__.py': '', 'src/bmts/common/constants.py': '"""Project-wide constants: MRI modalities and the canonical label scheme.\n\nCanonical labels (docs/PROJECT_STRUCTURE.md §3.2): 0 background, 1 NCR/NET, 2 ED, 3 ET, 4 RC (post-treatment only).\nThe cleaning step (M1-T04) writes every mask in this scheme, e.g. BraTS 2021\'s ET label 4 becomes 3.\n"""\n\nMODALITIES = ("t1", "t1ce", "t2", "flair")  # channel order of every model input\nSEG = "seg"\n\nBACKGROUND, NCR, ED, ET, RC = 0, 1, 2, 3, 4\n\n# Evaluation regions, in the channel order of the model output. RC is never part of a tumor region.\nREGIONS = {\n    "TC": (NCR, ET),       # tumor core\n    "WT": (NCR, ED, ET),   # whole tumor\n    "ET": (ET,),           # enhancing tumor\n}\n', 'src/bmts/data/__init__.py': '', 'src/bmts/data/augmentation/__init__.py': '', 'src/bmts/data/augmentation/transforms.py': '"""MONAI transform chains for training and validation (M1-T05).\n\nInput: one cleaned case = {t1, t1ce, t2, flair, seg} file paths (output of M1-T04, already RAS,\n240x240x155, z-scored per modality, masks in the canonical label scheme).\nOutput: "image" (4, H, W, D) float32 in MODALITIES order, "label" (3, H, W, D) float32 region masks (TC, WT, ET).\n"""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nfrom monai import transforms as T\nfrom monai.data import MetaTensor\n\nfrom bmts.common.constants import MODALITIES, REGIONS, SEG\n\nFOREGROUND = "fg"  # temporary whole-tumor mask used only to pick training patches\n\n\ndef nonzero(x):\n    """Brain voxels: the cleaned scans are z-scored, so the brain has negative values too; background is exactly 0."""\n    return x != 0\n\n\nclass SegToRegionsd(T.MapTransform):\n    """Canonical label map (1 channel) -> one binary channel per region in REGIONS (TC, WT, ET).\n\n    Also writes a 1-channel whole-tumor mask under `foreground_key` for tumor-centred patch sampling.\n    Labels outside every region (background, resection cavity 4) end up 0 in every channel.\n    """\n\n    def __init__(self, keys=SEG, label_key="label", foreground_key=FOREGROUND, regions=REGIONS):\n        super().__init__(keys)\n        self.label_key, self.foreground_key = label_key, foreground_key\n        self.regions = {name: torch.tensor(labels) for name, labels in regions.items()}\n\n    def __call__(self, data):\n        d = dict(data)\n        for key in self.key_iterator(d):\n            seg = torch.as_tensor(d[key])[0].round().long()\n            channels = torch.stack([torch.isin(seg, labels) for labels in self.regions.values()]).float()\n            whole = channels[list(self.regions).index("WT")][None] if "WT" in self.regions else channels.amax(0, keepdim=True)\n            if isinstance(d[key], MetaTensor):\n                channels = MetaTensor(channels, meta=d[key].meta)\n                whole = MetaTensor(whole, meta=d[key].meta)\n            d[self.label_key], d[self.foreground_key] = channels, whole\n            del d[key]\n        return d\n\n\ndef load_transforms(normalize=False):\n    """Read the 5 files of a case, stack the 4 MRI types as channels, turn the mask into region channels."""\n    keys = [*MODALITIES, SEG]\n    chain = [\n        T.LoadImaged(keys, image_only=True),\n        T.EnsureChannelFirstd(keys),\n        T.ConcatItemsd(list(MODALITIES), "image"),\n        T.DeleteItemsd(list(MODALITIES)),\n        T.Orientationd(["image", SEG], axcodes="RAS", labels=(("L", "R"), ("P", "A"), ("I", "S"))),\n        T.EnsureTyped("image", dtype=torch.float32),\n    ]\n    if normalize:  # only for scans that were not z-scored by the cleaning step\n        chain.append(T.NormalizeIntensityd("image", nonzero=True, channel_wise=True))\n    chain.append(SegToRegionsd(SEG))\n    return chain\n\n\ndef train_transforms(cfg):\n    """Loading + random tumor-centred patch + augmentation (flips, small rotation/scaling, intensity, gamma)."""\n    roi = cfg["roi_size"]\n    aug, sampling = cfg["augment"], cfg["sampling"]\n    keys = ["image", "label", FOREGROUND]\n    chain = load_transforms(cfg.get("normalize", False)) + [\n        T.CropForegroundd(keys, source_key="image", select_fn=nonzero, allow_smaller=True),\n        T.SpatialPadd(keys, spatial_size=roi),\n        T.RandCropByPosNegLabeld(["image", "label"], label_key=FOREGROUND, spatial_size=roi,\n                                 pos=sampling["pos"], neg=sampling["neg"], num_samples=1),\n        T.DeleteItemsd(FOREGROUND),\n    ]\n    for axis in range(3):\n        chain.append(T.RandFlipd(["image", "label"], prob=aug["flip_prob"], spatial_axis=axis))\n    if aug["affine_prob"] > 0:\n        angle = math.radians(aug["rotate_deg"])\n        chain.append(T.RandAffined(["image", "label"], prob=aug["affine_prob"], rotate_range=(angle,) * 3,\n                                   scale_range=(aug["scale"],) * 3, mode=("bilinear", "nearest"),\n                                   padding_mode="zeros"))\n    if aug.get("elastic_prob", 0) > 0:\n        chain.append(T.Rand3DElasticd(["image", "label"], sigma_range=(5, 8), magnitude_range=(100, 200),\n                                      prob=aug["elastic_prob"], mode=("bilinear", "nearest")))\n    chain += [\n        T.RandScaleIntensityd("image", factors=0.1, prob=aug["intensity_prob"]),\n        T.RandShiftIntensityd("image", offsets=0.1, prob=aug["intensity_prob"]),\n        T.RandAdjustContrastd("image", gamma=(0.7, 1.5), prob=aug["gamma_prob"]),\n        T.EnsureTyped(["image", "label"], dtype=torch.float32, track_meta=False),\n    ]\n    return T.Compose(chain)\n\n\ndef val_transforms(cfg):\n    """Loading only: the whole 240x240x155 volume, for sliding-window evaluation."""\n    return T.Compose(load_transforms(cfg.get("normalize", False)) + [\n        T.DeleteItemsd(FOREGROUND),\n        T.EnsureTyped(["image", "label"], dtype=torch.float32, track_meta=False),\n    ])\n', 'src/bmts/data/datasets/__init__.py': '', 'src/bmts/data/datasets/brats.py': '"""Find the cleaned cases of a split and build the training / validation data loaders (M1-T05).\n\nCleaned layout (M1-T04, Yasmin\'s pipeline/standardize.py):\n    <processed_root>/<dataset>/<subject>/<timepoint>/{t1,t1ce,t2,flair,seg}.nii.gz\nSplit file (M1-T04, qc/split_v1.csv): one row per patient with dataset_source, canonical_subject_id, split, lockbox.\n\nLocked patients (split "test" or lockbox True) are refused unless allow_lockbox=True, which only the frozen\nfinal evaluation (M5-T02) may set.\n"""\nfrom __future__ import annotations\n\nimport warnings\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nimport yaml\nfrom monai.data import CacheDataset, DataLoader, Dataset\n\nfrom bmts.common.constants import MODALITIES, SEG\nfrom bmts.data.augmentation.transforms import train_transforms, val_transforms\n\nREQUIRED_COLUMNS = {"dataset_source", "canonical_subject_id", "split"}\n\n\nclass LockboxError(RuntimeError):\n    """Raised when code asks for locked test patients outside the final evaluation."""\n\n\ndef load_config(path, **overrides):\n    """Read a loader YAML config; keyword overrides replace top-level keys (None values are ignored)."""\n    cfg = yaml.safe_load(Path(path).read_text(encoding="utf-8"))\n    cfg.update({k: v for k, v in overrides.items() if v is not None})\n    return cfg\n\n\ndef list_cases(split_csv, processed_root, splits=("train",), datasets=("brats2021",), limit=None,\n               allow_lockbox=False, on_missing="error"):\n    """Return one dict per (patient, timepoint): {"id", "t1", "t1ce", "t2", "flair", "seg"} file paths.\n\n    limit keeps the first N patients (sorted by ID) before the files are checked, for quick smoke tests.\n    on_missing: "error" stops on the first incomplete case list; "skip" drops those cases with a warning.\n    """\n    table = pd.read_csv(split_csv, dtype=str, keep_default_na=False)\n    missing_columns = REQUIRED_COLUMNS - set(table.columns)\n    if missing_columns:\n        raise ValueError(f"{split_csv} lacks columns {sorted(missing_columns)}")\n    rows = table[table.split.isin(splits) & table.dataset_source.isin(datasets)]\n    locked = rows.split.eq("test")\n    if "lockbox" in rows:\n        locked |= rows.lockbox.str.lower().isin(["true", "1", "yes"])\n    if locked.any() and not allow_lockbox:\n        raise LockboxError(f"{int(locked.sum())} requested patients are in the locked test set; "\n                           "they may only be loaded by the final evaluation (M5-T02).")\n    rows = rows.sort_values("canonical_subject_id")\n    if limit is not None:\n        rows = rows.head(limit)\n\n    cases, problems = [], []\n    for row in rows.itertuples():\n        subject = row.canonical_subject_id.split("__", 1)[-1]\n        subject_dir = Path(processed_root) / row.dataset_source / subject\n        timepoints = sorted(p for p in subject_dir.iterdir() if p.is_dir()) if subject_dir.is_dir() else []\n        if not timepoints:\n            problems.append(f"{row.canonical_subject_id}: no cleaned scans in {subject_dir}")\n            continue\n        for tp in timepoints:\n            files = {k: tp / f"{k}.nii.gz" for k in (*MODALITIES, SEG)}\n            absent = [k for k, p in files.items() if not p.is_file()]\n            if absent:\n                problems.append(f"{row.canonical_subject_id}/{tp.name}: missing {\', \'.join(absent)}")\n                continue\n            cases.append({"id": f"{row.canonical_subject_id}/{tp.name}", **{k: str(p) for k, p in files.items()}})\n\n    if problems:\n        message = f"{len(problems)} of {len(rows)} patients are incomplete, e.g.\\n  " + "\\n  ".join(problems[:5])\n        if on_missing == "error":\n            raise FileNotFoundError(message + "\\nRun the cleaning step for them, or set on_missing: skip.")\n        warnings.warn(message, stacklevel=2)\n    return cases\n\n\ndef build_loaders(cfg, train_limit=None, val_limit=None):\n    """Training loader (random augmented patches) and validation loader (whole volumes, batch size 1)."""\n    common = dict(split_csv=cfg["split_csv"], processed_root=cfg["processed_root"], datasets=cfg["datasets"],\n                  on_missing=cfg.get("on_missing", "error"))\n    train_cases = list_cases(splits=(cfg["train_split"],), limit=train_limit, **common)\n    val_cases = list_cases(splits=(cfg["val_split"],), limit=val_limit, **common)\n    if not train_cases:\n        raise RuntimeError("No training cases found: check split_csv, processed_root and datasets in the config.")\n\n    cache_rate = cfg.get("cache_rate", 0.0)\n    workers = cfg.get("num_workers", 0)\n    if cache_rate > 0:\n        train_ds = CacheDataset(train_cases, train_transforms(cfg), cache_rate=cache_rate, num_workers=workers)\n    else:\n        train_ds = Dataset(train_cases, train_transforms(cfg))\n    val_ds = Dataset(val_cases, val_transforms(cfg))\n    extra = dict(num_workers=workers, pin_memory=torch.cuda.is_available(), persistent_workers=workers > 0)\n    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=False, **extra)\n    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, **extra)\n    return train_loader, val_loader\n', 'configs/data/loader_v1.yaml': '# Data loader settings (M1-T05). Read by bmts.data.datasets.brats.load_config / build_loaders.\n# Paths are for Colab; override them on the command line elsewhere (scripts/smoke_train.py --help).\n\nprocessed_root: /content/processed                          # cleaned scans: <dataset>/<subject>/<timepoint>/{t1,t1ce,t2,flair,seg}.nii.gz\nsplit_csv: /content/drive/MyDrive/BrainMRI_Data/qc/split_v1.csv   # patient-level split from M1-T04\ndatasets: [brats2021]        # the main model trains on BraTS 2021 only (docs/dataset_research.md §8.2, rule 4)\ntrain_split: train\nval_split: val\non_missing: error            # error | skip (skip drops cases whose cleaned files are missing, with a warning)\nnormalize: false             # cleaned scans are already z-scored per MRI type; true only for uncleaned scans\n\nroi_size: [128, 128, 128]    # training patch in voxels (1 mm); use [96, 96, 96] on GPUs with less than 8 GB\nbatch_size: 2\nnum_workers: 2               # Colab has 2 CPU cores\ncache_rate: 0.0              # share of training cases kept in RAM; one case is about 150 MB, Colab has about 12 GB\n\nsampling:                    # training patches centred on tumor (pos) versus on other brain tissue (neg)\n  pos: 2\n  neg: 1\n\naugment:                     # proposal Milestone 1: rotations, flips, scaling, intensity/gamma shifts, elastic\n  flip_prob: 0.5             # per axis\n  affine_prob: 0.2           # small rotation + scaling\n  rotate_deg: 15\n  scale: 0.1\n  intensity_prob: 0.5        # brightness scale and shift\n  gamma_prob: 0.15\n  elastic_prob: 0.0          # elastic deformation is slow on CPU; switch on in a later experiment if useful\n\nseed: 42\n', 'scripts/smoke_train.py': '"""M1-T05 smoke test: prove the data loader feeds a model without errors (2 tiny epochs).\n\n    python scripts/smoke_train.py --config configs/data/loader_v1.yaml --cases 8 --epochs 2\n\nIt trains a deliberately small 3D U-Net on a few training patients, runs one validation pass on a few\nvalidation patients, and writes a JSON report. The scores mean nothing after 2 epochs: the point is that\nevery batch loads with the right shapes and values. Locked test patients are never read.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\n\nimport torch  # noqa: E402\nimport mlflow  # noqa: E402\nfrom monai.inferers import sliding_window_inference  # noqa: E402\nfrom monai.losses import DiceLoss  # noqa: E402\nfrom monai.metrics import DiceMetric  # noqa: E402\nfrom monai.networks.nets import UNet  # noqa: E402\nfrom monai.utils import set_determinism  # noqa: E402\n\nfrom bmts.common.constants import MODALITIES, REGIONS  # noqa: E402\nfrom bmts.data.datasets.brats import build_loaders, load_config  # noqa: E402\n\n\ndef check_batch(batch, roi):\n    """Fail loudly if a training batch does not have the shapes and values the model expects."""\n    image, label = batch["image"], batch["label"]\n    assert image.ndim == 5 and image.shape[1] == len(MODALITIES), f"image shape {tuple(image.shape)}"\n    assert label.shape[1] == len(REGIONS) and label.shape[2:] == image.shape[2:], f"label shape {tuple(label.shape)}"\n    assert list(image.shape[2:]) == list(roi), f"patch {tuple(image.shape[2:])} != roi {roi}"\n    assert torch.isfinite(image).all(), "image has NaN/Inf"\n    assert set(label.unique().tolist()) <= {0.0, 1.0}, "label is not binary"\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    p.add_argument("--config", default="configs/data/loader_v1.yaml")\n    p.add_argument("--processed-root", help="override processed_root")\n    p.add_argument("--split-csv", help="override split_csv")\n    p.add_argument("--cases", type=int, default=8, help="training patients to use")\n    p.add_argument("--val-cases", type=int, default=2, help="validation patients to use")\n    p.add_argument("--epochs", type=int, default=2)\n    p.add_argument("--roi", type=int, nargs=3, help="patch size, e.g. 96 96 96")\n    p.add_argument("--batch-size", type=int)\n    p.add_argument("--num-workers", type=int)\n    p.add_argument("--report", default="smoke_report.json")\n    p.add_argument("--mlflow-uri", default=None, help="MLflow tracking URI")\n    p.add_argument("--mlflow-experiment", default="BrainMRI_3D_UNet", help="MLflow experiment name")\n    p.add_argument("--mlflow-run-name", default=None, help="Optional MLflow run name")\n    args = p.parse_args()\n    report_path = Path(args.report)\n    if args.mlflow_uri:\n        mlflow.set_tracking_uri(args.mlflow_uri)\n    mlflow.set_experiment(args.mlflow_experiment)\n\n    cfg = load_config(args.config, processed_root=args.processed_root, split_csv=args.split_csv,\n                      roi_size=args.roi, batch_size=args.batch_size, num_workers=args.num_workers)\n    with mlflow.start_run(run_name=args.mlflow_run_name) as run:\n        set_determinism(cfg.get("seed", 42))\n        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        amp = device.type == "cuda"\n        print(f"device: {device}{\' (\' + torch.cuda.get_device_name(0) + \')\' if amp else \'\'}")\n        mlflow.log_params({\n            "train_cases": args.cases,\n            "val_cases": args.val_cases,\n            "epochs": args.epochs,\n            "batch_size": cfg["batch_size"],\n            "roi_size": str(cfg["roi_size"]),\n            "num_workers": cfg.get("num_workers", 0),\n            "learning_rate": 1e-3,\n            "model": "MONAI 3D UNet",\n            "modalities": str(MODALITIES),\n            "regions": str(tuple(REGIONS.keys())),\n            "split_csv": str(cfg["split_csv"]),\n            "processed_root": str(cfg["processed_root"]),\n        })\n\n        train_loader, val_loader = build_loaders(cfg, train_limit=args.cases, val_limit=args.val_cases)\n        print(f"training cases: {len(train_loader.dataset)}, validation cases: {len(val_loader.dataset)}, "\n              f"patch {cfg[\'roi_size\']}, batch {cfg[\'batch_size\']}")\n\n        model = UNet(spatial_dims=3, in_channels=len(MODALITIES), out_channels=len(REGIONS),\n                     channels=(16, 32, 64, 128), strides=(2, 2, 2), num_res_units=1).to(device)\n        loss_fn = DiceLoss(sigmoid=True, smooth_nr=1e-5, smooth_dr=1e-5)\n        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n        scaler = torch.amp.GradScaler("cuda", enabled=amp)\n        report = {"device": str(device), "config": cfg, "epochs": []}\n\n        for epoch in range(1, args.epochs + 1):\n            model.train()\n            start, losses = time.time(), []\n            for step, batch in enumerate(train_loader):\n                check_batch(batch, cfg["roi_size"])\n                if epoch == 1 and step == 0:\n                    img, lab = batch["image"], batch["label"]\n                    print(f"first batch: image {tuple(img.shape)} {img.dtype}, label {tuple(lab.shape)}, "\n                          f"tumor voxels per region {[int(v) for v in lab.sum(dim=(0, 2, 3, 4))]}")\n                image, label = batch["image"].to(device), batch["label"].to(device)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device.type, enabled=amp):\n                    loss = loss_fn(model(image), label)\n                assert torch.isfinite(loss), f"loss is {loss.item()}"\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n                losses.append(loss.item())\n            train_time = time.time() - start\n\n            model.eval()\n            dice = DiceMetric(include_background=True, reduction="mean_batch")\n            start = time.time()\n            with torch.no_grad(), torch.autocast(device.type, enabled=amp):\n                for batch in val_loader:\n                    image, label = batch["image"].to(device), batch["label"].to(device)\n                    logits = sliding_window_inference(image, cfg["roi_size"], 1, model, overlap=0.25)\n                    dice((torch.sigmoid(logits.float()) > 0.5).float(), label)\n            scores = dice.aggregate().tolist() if len(val_loader.dataset) else []\n            entry = {"epoch": epoch, "train_batches": len(losses), "mean_loss": sum(losses) / len(losses),\n                     "train_seconds": round(train_time, 1), "val_seconds": round(time.time() - start, 1),\n                     "val_dice": dict(zip(REGIONS, [round(s, 4) for s in scores]))}\n            report["epochs"].append(entry)\n            mlflow.log_metric("train_loss", entry["mean_loss"], step=epoch)\n            for region, score in entry["val_dice"].items():\n                mlflow.log_metric(f"val_dice_{region}", score, step=epoch)\n            print(f"epoch {epoch}: {entry[\'train_batches\']} batches, loss {entry[\'mean_loss\']:.4f}, "\n                  f"{entry[\'train_seconds\']}s train, {entry[\'val_seconds\']}s val, val Dice {entry[\'val_dice\']}")\n\n        if amp:\n            report["peak_gpu_memory_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n            print(f"peak GPU memory: {report[\'peak_gpu_memory_gb\']} GB")\n        report["result"] = "passed"\n        report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")\n\n        checkpoint_path = report_path.with_name("smoke_model.pt")\n        torch.save(model.state_dict(), checkpoint_path)\n\n        mlflow.log_artifact(str(report_path), artifact_path="reports")\n        mlflow.log_artifact(str(checkpoint_path), artifact_path="model")\n        mlflow.set_tag("status", "passed")\n        mlflow.set_tag("device", str(device))\n\n        print(f"MLflow run: {run.info.run_id}")\n        print(f"SMOKE TEST PASSED: every batch loaded with the right shapes and values. Report: {report_path}")\n\nif __name__ == "__main__":\n    main()\n'}

for rel, content in FILES.items():
    path = CODE / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

print(f'wrote {len(FILES)} project files to {CODE}')

wrote 10 project files to /content/repo


In [ ]:
# 5A. Prepare the exact smoke-test patients with the existing M1-T04 pipeline
# This is the cleaning step from the original 03_dataloader_smoke_test notebook.
# It is required because the DataLoader expects cleaned files under /content/processed/brats2021/.
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(BASE / 'pipeline'))
import standardize as S

split = pd.read_csv(SPLIT_CSV, dtype=str)
split = split[split.dataset_source == 'brats2021'].sort_values('canonical_subject_id')

def first(group, n):
    return split[split.split == group].canonical_subject_id.str.split('__').str[1].head(n).tolist()

CASES, VAL_CASES = 16, 4
subjects = first('train', CASES) + first('val', VAL_CASES)
print(f'Selected {len(subjects)} patients: {subjects[:3]} ...')

S.ensure_extracted('brats2021')
jobs = S.run(['brats2021'], subjects=set(subjects))

failed = jobs[jobs.status == 'failed']
if not failed.empty:
    display(failed[['subject', 'scan', 'error']])
    raise RuntimeError('The cleaning step failed for one or more selected patients.')

# Verify that the folders expected by the DataLoader now exist.
processed_root = Path('/content/processed/brats2021')
missing = []
for subject in subjects:
    subject_dir = processed_root / subject
    if not subject_dir.exists():
        missing.append(str(subject_dir))

print('Processed root exists:', processed_root.exists())
print('Missing processed patient folders:', len(missing))
if missing:
    print('\n'.join(missing[:10]))
    raise FileNotFoundError('Cleaning completed, but the expected processed patient folders were not created.')


Selected 20 patients: ['BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003'] ...


brats2021:   0%|          | 0.00/13.4G [00:00<?, ?B/s]

  0%|          | 0/100 [00:00<?, ?it/s]

status
done    100
Processed root exists: True
Missing processed patient folders: 0


In [ ]:
# 5. Configure MLflow tracking with SQLite (supported by current MLflow)
import mlflow

# Current MLflow versions reject the old file:// tracking backend.
# SQLite keeps the experiment database persistent on Google Drive.
MLFLOW_DB = BASE / 'mlflow.db'
MLFLOW_URI = f'sqlite:///{MLFLOW_DB}'
EXPERIMENT = 'BrainMRI_3D_UNet'

mlflow.set_tracking_uri(MLFLOW_URI)

experiment = mlflow.get_experiment_by_name(EXPERIMENT)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT)
    experiment = mlflow.get_experiment(experiment_id)

print('MLflow tracking URI:', mlflow.get_tracking_uri())
print('Experiment name:', experiment.name)
print('Experiment ID:', experiment.experiment_id)


2026/10/01 16:10:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/01 16:10:47 INFO mlflow.store.db.utils: Updating database tables


MLflow tracking URI: sqlite:////content/drive/MyDrive/BrainMRI_Data/mlflow.db
Experiment name: BrainMRI_3D_UNet
Experiment ID: 1


## 5. What is now connected to MLflow

Inside the existing training script, one MLflow run is opened around the actual training. The run records:

- **Parameters:** number of training/validation cases, epochs, batch size, ROI size, learning rate, model name, modalities, regions, split file, and processed-data path.
- **Metrics per epoch:** training loss and Dice for TC/WT/ET.
- **Artifacts:** the JSON smoke-test report and `smoke_model.pt` containing the trained model weights.
- **Tags:** run status and device.

This is still a **smoke test**; the notebook deliberately keeps the existing 2-epoch setup rather than pretending this is the final model training.

In [ ]:
# 6B. Smoke-test settings
EPOCHS = 2
print('Training cases:', CASES)
print('Validation cases:', VAL_CASES)
print('Epochs:', EPOCHS)


Training cases: 16
Validation cases: 4
Epochs: 2


In [ ]:
# 7. Run the EXISTING smoke training code with MLflow enabled
%cd /content/repo

REPORT = OUT / 'smoke_report.json'

!python scripts/smoke_train.py \
  --config configs/data/loader_v1.yaml \
  --processed-root /content/processed \
  --split-csv "$SPLIT_CSV" \
  --cases $CASES \
  --val-cases $VAL_CASES \
  --epochs $EPOCHS \
  --report "$REPORT" \
  --mlflow-uri "$MLFLOW_URI" \
  --mlflow-experiment "$EXPERIMENT" \
  --mlflow-run-name "M1-T05-MLflow-smoke"


/content/repo
2026-10-01 16:11:35.045720: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-01 16:11:37.155439: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
device: cpu
training cases: 16, validation cases: 4, patch [128, 128, 128], batch 2
first batch: image (2, 4, 128, 128, 128) torch.float32, label (2, 3, 128, 128, 128), tumor voxels per region [40820, 291778, 29327]
epoch 1: 8 batches, loss 0.9353, 107.0s train, 84.9s val, val Dice {'TC': 0.012, 'WT': 0.0336, 'ET': 0.0066}
epoch 2: 8 batches, loss 0.9257, 105.0s train, 85.4s val, val Dice {'TC': 0.0295, 'WT': 0.0586, 'ET': 0.0133}
MLflow run: 215715ab9fa8422092a767175f00e04d
SMOKE TEST PASSED: every batch loaded with the

In [ ]:
# 8. Verify the outputs saved by the run
import json

report = json.loads(REPORT.read_text())
MODEL_FILE = REPORT.with_name('smoke_model.pt')

print('Result:', report['result'])
print('Device:', report['device'])
print('Report exists:', REPORT.exists())
print('Model file exists:', MODEL_FILE.exists())
print('MLflow database exists:', MLFLOW_DB.exists())
print('MLflow database:', MLFLOW_DB)

assert report['result'] == 'passed'
assert REPORT.exists()
assert MODEL_FILE.exists()
assert MLFLOW_DB.exists()


Result: passed
Device: cpu
Report exists: True
Model file exists: True
MLflow database exists: True
MLflow database: /content/drive/MyDrive/BrainMRI_Data/mlflow.db


In [ ]:
# 9. Inspect the MLflow experiment and the latest run
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=MLFLOW_URI)
experiment = client.get_experiment_by_name(EXPERIMENT)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['attributes.start_time DESC'],
    max_results=5,
)

print('Experiment:', experiment.name)
print('Number of recent runs:', len(runs))

for r in runs:
    print('\nRun ID:', r.info.run_id)
    print('Status:', r.info.status)
    print('Params:', r.data.params)
    print('Metrics:', r.data.metrics)
    artifacts = client.list_artifacts(r.info.run_id)
    print('Artifacts:', [a.path for a in artifacts])


Experiment: BrainMRI_3D_UNet
Number of recent runs: 1

Run ID: 215715ab9fa8422092a767175f00e04d
Status: FINISHED
Params: {'train_cases': '16', 'val_cases': '4', 'epochs': '2', 'batch_size': '2', 'roi_size': '[128, 128, 128]', 'num_workers': '2', 'learning_rate': '0.001', 'model': 'MONAI 3D UNet', 'modalities': "('t1', 't1ce', 't2', 'flair')", 'regions': "('TC', 'WT', 'ET')", 'split_csv': '/content/drive/MyDrive/BrainMRI_Data/qc/split_v1.csv', 'processed_root': '/content/processed'}
Metrics: {'train_loss': 0.9257170185446739, 'val_dice_TC': 0.0295, 'val_dice_WT': 0.0586, 'val_dice_ET': 0.0133}
Artifacts: ['model', 'reports']


In [ ]:
# Computes the Dice score between two binary masks (predicted vs ground truth).
import numpy as np

def dice_score(pred: np.ndarray, target: np.ndarray) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)

    intersection = np.logical_and(pred, target).sum()
    total = pred.sum() + target.sum()

    if total == 0:
        # both masks are empty: perfect agreement
        return 1.0

    return 2.0 * intersection / total

In [ ]:
# Computes the 95th-percentile Hausdorff Distance (HD95) between two binary masks, in millimeters.
import numpy as np
from scipy.ndimage import distance_transform_edt, binary_erosion

def surface_distances(mask: np.ndarray, spacing: tuple) -> np.ndarray:
    """Distance (in mm) from every surface voxel of `mask` to the nearest True voxel."""
    eroded = binary_erosion(mask)
    surface = mask & ~eroded          # outer shell of the mask, 1 voxel thick
    distance_map = distance_transform_edt(~surface, sampling=spacing)
    return distance_map


def hd95(pred: np.ndarray, target: np.ndarray, spacing: tuple = (1.0, 1.0, 1.0)) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)

    if not pred.any() and not target.any():
        return 0.0   # both empty: perfect agreement, zero distance
    if not pred.any() or not target.any():
        return float('nan')   # one empty, the other not: distance is undefined, matches MONAI's convention

    pred_surface = pred & ~binary_erosion(pred)
    target_surface = target & ~binary_erosion(target)

    dist_to_target = distance_transform_edt(~target, sampling=spacing)
    dist_to_pred = distance_transform_edt(~pred, sampling=spacing)

    distances_pred_to_target = dist_to_target[pred_surface]
    distances_target_to_pred = dist_to_pred[target_surface]

    all_distances = np.concatenate([distances_pred_to_target, distances_target_to_pred])
    return float(np.percentile(all_distances, 95))

In [ ]:
# Computes Dice and HD95 for all three tumor regions (WT, TC, ET) from a full segmentation map.
NCR, ED, ET_LABEL = 1, 2, 3
REGIONS = {
    'WT': (NCR, ED, ET_LABEL),   # whole tumor: everything that is not background
    'TC': (NCR, ET_LABEL),       # tumor core: necrotic + enhancing, excludes edema
    'ET': (ET_LABEL,),           # enhancing tumor only
}

def region_mask(seg: np.ndarray, labels: tuple) -> np.ndarray:
    """True wherever seg has one of the given label values."""
    return np.isin(seg, labels)


def evaluate_case(pred_seg: np.ndarray, target_seg: np.ndarray, spacing: tuple = (1.0, 1.0, 1.0)) -> dict:
    results = {}
    for region_name, labels in REGIONS.items():
        pred_mask = region_mask(pred_seg, labels)
        target_mask = region_mask(target_seg, labels)
        results[region_name] = {
            'dice': dice_score(pred_mask, target_mask),
            'hd95': hd95(pred_mask, target_mask, spacing=spacing),
        }
    return results

In [ ]:
# Compares our Dice and HD95 against MONAI's on the same synthetic example.
import torch
# Installs MONAI in this notebook's environment.
!pip install -q monai
from monai.metrics import DiceMetric, HausdorffDistanceMetric

dice_metric = DiceMetric(include_background=False, reduction='mean')
hd_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction='mean')

for region_name, labels in REGIONS.items():
    pred_mask = region_mask(seg_pred, labels)
    target_mask = region_mask(seg_true, labels)

    # MONAI expects (batch, channel, *spatial) tensors
    pred_t = torch.as_tensor(pred_mask, dtype=torch.float32)[None, None]
    target_t = torch.as_tensor(target_mask, dtype=torch.float32)[None, None]

    monai_dice = dice_metric(pred_t, target_t).item()
    monai_hd95 = hd_metric(pred_t, target_t).item()

    ours = evaluate_case(seg_pred, seg_true)[region_name]
    print(f'{region_name}: ours dice={ours["dice"]:.4f} hd95={ours["hd95"]:.4f}  |  '
          f'MONAI dice={monai_dice:.4f} hd95={monai_hd95:.4f}')

WT: ours dice=1.0000 hd95=0.0000  |  MONAI dice=1.0000 hd95=0.0000
TC: ours dice=1.0000 hd95=0.0000  |  MONAI dice=1.0000 hd95=0.0000
ET: ours dice=0.5000 hd95=1.0000  |  MONAI dice=0.5000 hd95=1.0000


In [ ]:
# Edge case: predicted mask is completely empty for one region, target is not.
seg_true2 = np.zeros((10, 10, 10), dtype=int)
seg_true2[2:6, 2:6, 2:6] = 1
seg_true2[3:5, 3:5, 3:5] = 3

seg_pred2 = np.zeros((10, 10, 10), dtype=int)
seg_pred2[2:6, 2:6, 2:6] = 1
# no label 3 at all in the prediction: ET mask is empty

ours2 = evaluate_case(seg_pred2, seg_true2)
print('ours:', ours2['ET'])

pred_mask2 = region_mask(seg_pred2, (3,))
target_mask2 = region_mask(seg_true2, (3,))
pred_t2 = torch.as_tensor(pred_mask2, dtype=torch.float32)[None, None]
target_t2 = torch.as_tensor(target_mask2, dtype=torch.float32)[None, None]
print('MONAI dice:', dice_metric(pred_t2, target_t2).item())
print('MONAI hd95:', hd_metric(pred_t2, target_t2).item())

ours: {'dice': np.float64(0.0), 'hd95': nan}
MONAI dice: 0.0
MONAI hd95: nan


In [ ]:
# Extracts and standardizes ONE BraTS2021 patient only, for quick testing (no full tar extraction needed).
from google.colab import drive
drive.mount('/content/drive')

import sys, tarfile
from pathlib import Path
sys.path.insert(0, '/content/drive/MyDrive/BrainMRI_Data/pipeline')
import standardize as sm
import pandas as pd

subject = 'BraTS2021_00000'
raw_dir = Path('/content/data/brats2021')
tar_path = sm.TARS['brats2021']

with tarfile.open(tar_path, 'r|') as tar:
    for member in tar:
        if f'{subject}/' in member.name and member.name.endswith('.nii.gz'):
            tar.extract(member, raw_dir, filter='data')
        elif raw_dir.exists() and (raw_dir / subject).exists() and f'{subject}/' not in member.name:
            # stop scanning once we've passed this patient's files
            if len(list((raw_dir / subject).glob('*.nii.gz'))) == 5:
                break

jobs = sm.list_jobs(['brats2021'])
jobs = jobs[jobs['subject'] == subject]
for row in jobs.itertuples():
    status, error = sm.standardize_file(row)
    print(status, row.scan, error)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
done t1 
done t1ce 
done t2 
done flair 
done seg 


In [ ]:
# Tests our functions on a real BraTS2021 segmentation file, with a small simulated prediction error.
import nibabel as nib
from pathlib import Path

seg_path = sorted(Path('/content/processed/brats2021').glob('*/0/seg.nii.gz'))[0]
print('Using:', seg_path)

seg_true_real = np.asarray(nib.load(seg_path).dataobj).astype(int)
print('Labels present:', sorted(np.unique(seg_true_real).tolist()))

# Simulate an imperfect prediction: erase a chunk of the enhancing tumor (label 3) only.
seg_pred_real = seg_true_real.copy()
et_voxels = np.argwhere(seg_pred_real == 3)
if len(et_voxels) > 0:
    center = et_voxels[len(et_voxels) // 2]
    z, y, x = center
    seg_pred_real[z - 3:z + 3, y - 3:y + 3, x - 3:x + 3] = 0   # erase a small cube inside the enhancing region

spacing = (1.0, 1.0, 1.0)   # all cleaned volumes share this voxel spacing

ours_real = evaluate_case(seg_pred_real, seg_true_real, spacing=spacing)

for region_name, labels in REGIONS.items():
    pred_mask = region_mask(seg_pred_real, labels)
    target_mask = region_mask(seg_true_real, labels)
    pred_t = torch.as_tensor(pred_mask, dtype=torch.float32)[None, None]
    target_t = torch.as_tensor(target_mask, dtype=torch.float32)[None, None]
    monai_dice = dice_metric(pred_t, target_t).item()
    monai_hd95 = hd_metric(pred_t, target_t).item()
    ours = ours_real[region_name]
    print(f'{region_name}: ours dice={ours["dice"]:.4f} hd95={ours["hd95"]:.4f}  |  '
          f'MONAI dice={monai_dice:.4f} hd95={monai_hd95:.4f}')

Using: /content/processed/brats2021/BraTS2021_00000/0/seg.nii.gz
Labels present: [0, 1, 2, 3]


/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:220: FutureWarning: monai.metrics.utils get_mask_edges:always_return_as_numpy: Argument `always_return_as_numpy` has been deprecated since version 1.5.0. It will be removed in version 1.7.0. The option is removed and the return type will always be equal to the input type.
  warn_deprecated(argname, msg, warning_category)


WT: ours dice=0.9981 hd95=0.0000  |  MONAI dice=0.9981 hd95=0.0000
TC: ours dice=0.9976 hd95=0.0000  |  MONAI dice=0.9976 hd95=0.0000
ET: ours dice=0.9969 hd95=0.0000  |  MONAI dice=0.9969 hd95=0.0000


In [ ]:
# Second test: erase a chunk at the EDGE of the tumor (not the center), to check HD95 reacts correctly.
seg_pred_edge = seg_true_real.copy()
wt_voxels = np.argwhere(seg_pred_edge > 0)
edge_voxel = wt_voxels[np.argmax(wt_voxels[:, 0])]   # a voxel at the extreme edge of the tumor (max z)
z, y, x = edge_voxel
seg_pred_edge[max(z - 4, 0):z + 1, y - 4:y + 4, x - 4:x + 4] = 0   # erase a small cube at the boundary

ours_edge = evaluate_case(seg_pred_edge, seg_true_real, spacing=spacing)

for region_name, labels in REGIONS.items():
    pred_mask = region_mask(seg_pred_edge, labels)
    target_mask = region_mask(seg_true_real, labels)
    pred_t = torch.as_tensor(pred_mask, dtype=torch.float32)[None, None]
    target_t = torch.as_tensor(target_mask, dtype=torch.float32)[None, None]
    monai_dice = dice_metric(pred_t, target_t).item()
    monai_hd95 = hd_metric(pred_t, target_t).item()
    ours = ours_edge[region_name]
    print(f'{region_name}: ours dice={ours["dice"]:.4f} hd95={ours["hd95"]:.4f}  |  '
          f'MONAI dice={monai_dice:.4f} hd95={monai_hd95:.4f}')

WT: ours dice=0.9982 hd95=0.0000  |  MONAI dice=0.9982 hd95=0.0000
TC: ours dice=0.9989 hd95=0.0000  |  MONAI dice=0.9989 hd95=0.0000
ET: ours dice=0.9986 hd95=0.0000  |  MONAI dice=0.9986 hd95=0.0000


In [ ]:
%%writefile /content/drive/MyDrive/BrainMRI_Data/pipeline/metrics.py
# Evaluation metrics (M2-T02): Dice and HD95 for WT, TC and ET, verified to match MONAI.
import numpy as np
from scipy.ndimage import distance_transform_edt, binary_erosion

NCR, ED, ET_LABEL = 1, 2, 3
REGIONS = {
    'WT': (NCR, ED, ET_LABEL),   # whole tumor: everything that is not background
    'TC': (NCR, ET_LABEL),       # tumor core: necrotic + enhancing, excludes edema
    'ET': (ET_LABEL,),           # enhancing tumor only
}


def dice_score(pred: np.ndarray, target: np.ndarray) -> float:
    """Dice similarity between two binary masks. Returns 1.0 if both are empty."""
    pred = pred.astype(bool)
    target = target.astype(bool)
    intersection = np.logical_and(pred, target).sum()
    total = pred.sum() + target.sum()
    if total == 0:
        return 1.0
    return 2.0 * intersection / total


def surface_distances(mask: np.ndarray, spacing: tuple) -> np.ndarray:
    """Distance (in mm) from every voxel to the nearest surface voxel of `mask`."""
    eroded = binary_erosion(mask)
    surface = mask & ~eroded
    return distance_transform_edt(~surface, sampling=spacing)


def hd95(pred: np.ndarray, target: np.ndarray, spacing: tuple = (1.0, 1.0, 1.0)) -> float:
    """95th-percentile Hausdorff Distance between two binary masks, in millimeters.

    Returns 0.0 if both masks are empty, nan if only one is empty (matches MONAI's convention).
    """
    pred = pred.astype(bool)
    target = target.astype(bool)
    if not pred.any() and not target.any():
        return 0.0
    if not pred.any() or not target.any():
        return float('nan')

    pred_surface = pred & ~binary_erosion(pred)
    target_surface = target & ~binary_erosion(target)
    dist_to_target = distance_transform_edt(~target, sampling=spacing)
    dist_to_pred = distance_transform_edt(~pred, sampling=spacing)
    distances_pred_to_target = dist_to_target[pred_surface]
    distances_target_to_pred = dist_to_pred[target_surface]
    all_distances = np.concatenate([distances_pred_to_target, distances_target_to_pred])
    return float(np.percentile(all_distances, 95))


def region_mask(seg: np.ndarray, labels: tuple) -> np.ndarray:
    """True wherever seg has one of the given label values."""
    return np.isin(seg, labels)


def evaluate_case(pred_seg: np.ndarray, target_seg: np.ndarray, spacing: tuple = (1.0, 1.0, 1.0)) -> dict:
    """Dice and HD95 for WT, TC and ET, from two full segmentation maps (not binary masks)."""
    results = {}
    for region_name, labels in REGIONS.items():
        pred_mask = region_mask(pred_seg, labels)
        target_mask = region_mask(target_seg, labels)
        results[region_name] = {
            'dice': dice_score(pred_mask, target_mask),
            'hd95': hd95(pred_mask, target_mask, spacing=spacing),
        }
    return results


def summarize_results(results_list: list) -> dict:
    """Averages Dice and HD95 over several cases (output of evaluate_case), ignoring nan values."""
    summary = {}
    for region_name in REGIONS:
        dice_values = [r[region_name]['dice'] for r in results_list]
        hd95_values = [r[region_name]['hd95'] for r in results_list]
        summary[region_name] = {
            'dice_mean': float(np.nanmean(dice_values)),
            'hd95_mean': float(np.nanmean(hd95_values)),
            'n_cases': len(results_list),
        }
    return summary

Writing /content/drive/MyDrive/BrainMRI_Data/pipeline/metrics.py


In [ ]:
# Quick check: import the saved metrics.py and test summarize_results on our two cases.
import importlib
import metrics as m
importlib.reload(m)

summary = m.summarize_results([ours_real, ours_edge])
import json
print(json.dumps(summary, indent=2))

{
  "WT": {
    "dice_mean": 0.998133683681437,
    "hd95_mean": 0.0,
    "n_cases": 2
  },
  "TC": {
    "dice_mean": 0.9982762935782536,
    "hd95_mean": 0.0,
    "n_cases": 2
  },
  "ET": {
    "dice_mean": 0.9977562908655707,
    "hd95_mean": 0.0,
    "n_cases": 2
  }
}
